# Phase 3.2 — SWDNet train từ đầu

Phase này bỏ checkpoint WDMamba/Phase 3 và khởi tạo **SWDNet bằng trọng số ngẫu nhiên**.
SWDNet là encoder–decoder Haar nhẹ với gated depthwise blocks, được viết mới cho thử nghiệm này;
không dùng teacher, KD hay pretrained weights. Mục tiêu là có một baseline kiến trúc mới ổn định,
đo được PSNR, SSIM, FPS và GFLOPs trên cùng protocol.

Notebook không tải ZIP dataset. `O-HAZY` là mặc định và lấy từ checkout VDT_GD2. SOTS chuẩn chỉ
đánh giá; muốn train từ đầu phải đặt `TRAIN_ROOT` riêng chứa ITS/OTS train và giữ SOTS test làm test.
Không dùng test để chọn checkpoint.

## 1. Chọn dataset và thiết lập runtime

Các giá trị hợp lệ: `reside6k`, `sots-its`, `sots-ots`, `o-hazy`, `i-haze`.
Kết quả nằm trong `/content/wdmamba_v3_2/<dataset>/<run_name>`.

In [ ]:
from pathlib import Path
import ast, json, os, platform, subprocess, sys

DATASET_NAME = 'o-hazy'  # @param ['reside6k', 'sots-its', 'sots-ots', 'o-hazy', 'i-haze']
DATA_ROOT_OVERRIDE = ''  # Dùng thư mục dataset đã có; trống = PROJECT_ROOT/dataset/... 
TRAIN_ROOT = ''          # Chỉ cần cho SOTS nếu muốn train ITS/OTS tách biệt.
EVAL_ONLY = None         # None tự nhận diện; False yêu cầu train/val; True chỉ test.
LOCAL_PROJECT = ''       # Checkout VDT_GD2 đã có trên Colab/Linux.
RUN_NAME = 'swdnet_v1'   # Đổi tên nếu muốn một lần train từ đầu khác.
RUN_MODE = 'train'       # train | resume | benchmark
EVAL_MAX_SIDE = 768
TRAIN_MAX_SIDE = 768
RUN_ROOT = Path('/content/wdmamba_v3_2') / DATASET_NAME / RUN_NAME
OUTPUT_ROOT = RUN_ROOT / 'results'
ENV_ROOT = Path('/content/wdmamba_o_hazy_env311')
PYTHON = str(ENV_ROOT/'bin/python') if (ENV_ROOT/'bin/python').is_file() else sys.executable
EVALUATION_CHECKPOINT = '' # SWDNet đã train nếu chỉ benchmark/cross-dataset SOTS.
PROJECT_URL = 'https://github.com/Darkin72/VDT_GD2.git'
SPEC = {'o-hazy': {'label': 'O-HAZY', 'relative_root': 'dataset/O-HAZY', 'kind': 'paired', 'can_train': True, 'val_ids': [5, 15, 25, 35]}, 'i-haze': {'label': 'I-HAZE', 'relative_root': 'dataset/I-HAZE', 'kind': 'paired', 'can_train': True, 'val_ids': []}, 'reside6k': {'label': 'RESIDE-6K', 'relative_root': 'dataset/RESIDE-6K', 'kind': 'paired', 'can_train': True, 'val_ids': []}, 'sots-its': {'label': 'SOTS Indoor / ITS', 'relative_root': 'dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]/indoor', 'kind': 'evaluation-only', 'can_train': False, 'val_ids': []}, 'sots-ots': {'label': 'SOTS Outdoor / OTS', 'relative_root': 'dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]/outdoor', 'kind': 'evaluation-only', 'can_train': False, 'val_ids': []}}[DATASET_NAME]
PROJECT_ROOT = Path(LOCAL_PROJECT).expanduser().resolve() if LOCAL_PROJECT else next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p/'dataset').is_dir()), Path('/content/VDT_GD2'))
if platform.system() != 'Linux':
    raise RuntimeError('Notebook cần Colab/Linux GPU; test CPU chạy test_cpu.py.')
RUN_ROOT.mkdir(parents=True, exist_ok=True); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def command(args, cwd=None, log='setup.log'):
    with (RUN_ROOT/log).open('a', encoding='utf-8') as out:
        proc = subprocess.Popen(list(map(str,args)), cwd=cwd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
        try:
            for line in proc.stdout:
                print(line, end='', flush=True); out.write(line); out.flush()
            rc = proc.wait()
        except BaseException:
            proc.kill(); proc.wait(); raise
    if rc: raise subprocess.CalledProcessError(rc, args)

if not DATA_ROOT_OVERRIDE and not PROJECT_ROOT.exists():
    command(['git','clone','--depth','1','--filter=blob:none','--sparse',PROJECT_URL,PROJECT_ROOT])
    command(['git','sparse-checkout','set','--skip-checks',SPEC['relative_root'],'solution/Haze Clear'], cwd=PROJECT_ROOT)
DATA_ROOT = Path(DATA_ROOT_OVERRIDE).expanduser().resolve() if DATA_ROOT_OVERRIDE else PROJECT_ROOT/SPEC['relative_root']
if not DATA_ROOT.is_dir() and not DATA_ROOT_OVERRIDE and (PROJECT_ROOT/'.git/info/sparse-checkout').is_file():
    command(['git','sparse-checkout','add','--skip-checks',SPEC['relative_root']], cwd=PROJECT_ROOT)
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Không có dataset {DATASET_NAME} tại {DATA_ROOT}. Đặt DATA_ROOT_OVERRIDE.')

CONFIG = {
    'dataset_name': DATASET_NAME, 'data_root': str(DATA_ROOT), 'project_root': str(PROJECT_ROOT),
    'train_root': TRAIN_ROOT, 'eval_only': EVAL_ONLY, 'work_root': str(RUN_ROOT),
    'output_root': str(OUTPUT_ROOT), 'seed': 2032, 'device': 'cuda',
    'model': {'width': 32, 'depths': [2,2,4,2,2]},
    'schedule': [{'name':'crop256','steps':10000,'crop':256,'batch_size':4},
                 {'name':'crop384','steps':4000,'crop':384,'batch_size':2},
                 {'name':'full','steps':2000,'crop':0,'batch_size':1}],
    'lr': 3e-4, 'min_lr': 1e-6, 'warmup_steps': 500, 'weight_decay': 1e-4,
    'ema_decay': .999, 'l1_weight': .05, 'edge_weight': .02, 'train_precision': 'bf16',
    'train_max_side': TRAIN_MAX_SIDE, 'eval_max_side': EVAL_MAX_SIDE, 'crop_border': 4,
    'cache_images': 40, 'val_every': 250, 'log_every': 100,
    'early_stop_patience': 0, 'early_stop_min_steps': 5000,
    'warmup': 10, 'timing_repeats': 50, 'evaluation_checkpoint': EVALUATION_CHECKPOINT,
}
(RUN_ROOT/'config.json').write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding='utf-8')
print('Dataset:', SPEC['label'], '| root:', DATA_ROOT, '| mode:', RUN_MODE)
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))


## 2. Mã tự chứa và kiểm tra CUDA

Các nguồn được ghi vào runtime trước khi chạy. Kiến trúc chỉ dùng PyTorch, không cần mamba-ssm.
Training BF16 chỉ bật khi GPU hỗ trợ; đánh giá và benchmark luôn FP32.

In [ ]:
# Reuse the old Python environment if present; otherwise use the Colab kernel Python.
dependency_probe = "import importlib.util,json; print(json.dumps([m for m in ['cv2','numpy','skimage'] if importlib.util.find_spec(m) is None]))"
missing = json.loads(subprocess.check_output([PYTHON, '-c', dependency_probe], text=True))
if missing:
    command([PYTHON, '-m', 'pip', 'install', 'numpy<2', 'opencv-python-headless<4.12', 'scikit-image'])
check = "import torch,cv2,numpy,skimage; assert torch.cuda.is_available(), 'Cần Colab GPU'; print('CUDA:',torch.cuda.get_device_name(),torch.__version__,torch.version.cuda)"
command([PYTHON, '-c', check])
freeze = subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True)
(OUTPUT_ROOT/'environment.txt').write_text(freeze, encoding='utf-8')


In [ ]:
DATA_SOURCE = '"""Paired-data and metric utilities for Phase 3.2; no pretrained weights."""\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport random\nimport re\n\nimport cv2\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom skimage.metrics import structural_similarity\n\nEXT = {\'.png\', \'.jpg\', \'.jpeg\', \'.bmp\', \'.tif\', \'.tiff\'}\n\n\ndef dump_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef write_csv(path, rows):\n    if not rows:\n        return\n    with Path(path).open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\nDATASET_SPECS = {\'o-hazy\': {\'label\': \'O-HAZY\',\n            \'relative_root\': \'dataset/O-HAZY\',\n            \'kind\': \'paired\',\n            \'can_train\': True,\n            \'val_ids\': [5, 15, 25, 35]},\n \'i-haze\': {\'label\': \'I-HAZE\',\n            \'relative_root\': \'dataset/I-HAZE\',\n            \'kind\': \'paired\',\n            \'can_train\': True,\n            \'val_ids\': []},\n \'reside6k\': {\'label\': \'RESIDE-6K\',\n              \'relative_root\': \'dataset/RESIDE-6K\',\n              \'kind\': \'paired\',\n              \'can_train\': True,\n              \'val_ids\': []},\n \'sots-its\': {\'label\': \'SOTS Indoor / ITS\',\n              \'relative_root\': \'dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]/indoor\',\n              \'kind\': \'evaluation-only\',\n              \'can_train\': False,\n              \'val_ids\': []},\n \'sots-ots\': {\'label\': \'SOTS Outdoor / OTS\',\n              \'relative_root\': \'dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]/outdoor\',\n              \'kind\': \'evaluation-only\',\n              \'can_train\': False,\n              \'val_ids\': []}}\n\n\ndef normalize_dataset_name(name):\n    key = str(name or \'o-hazy\').strip().lower().replace(\'_\', \'-\').replace(\'/\', \'-\').replace(\' \', \'-\')\n    key = re.sub(r\'-+\', \'-\', key).strip(\'-\')\n    aliases = {\'o-haze\': \'o-hazy\', \'ohazy\': \'o-hazy\', \'ihaze\': \'i-haze\', \'sots-indoor\': \'sots-its\', \'sots-indoor-its\': \'sots-its\',\n               \'its\': \'sots-its\', \'sots-outdoor\': \'sots-ots\', \'sots-outdoor-ots\': \'sots-ots\', \'ots\': \'sots-ots\', \'reside-6k\': \'reside6k\'}\n    key = aliases.get(key, key)\n    if key not in DATASET_SPECS:\n        raise ValueError(f\'DATASET_NAME không hợp lệ: {name}. Chọn một trong {list(DATASET_SPECS)}\')\n    return key\n\n\ndef _kind(path, root):\n    parents = {a.lower() for a in Path(path).relative_to(root).parts[:-1]}\n    stem = Path(path).stem.lower()\n    if parents & {\'gt\', \'clear\', \'clean\', \'groundtruth\', \'ground_truth\'} or stem.endswith((\'_gt\', \'_clear\', \'_clean\')):\n        return \'clear\'\n    if parents & {\'hazy\', \'haze\', \'input\', \'inputs\', \'hazy_images\'} or stem.endswith((\'_hazy\', \'_haze\', \'_input\')):\n        return \'hazy\'\n    return None\n\n\ndef _base_stem(path):\n    stem = Path(path).stem.lower()\n    stem = re.sub(r\'(?i)([_-]?hazy|[_-]?haze|[_-]?gt|[_-]?clear|[_-]?clean)$\', \'\', stem)\n    return re.sub(r\'\\d+\', lambda m: str(int(m[0])), stem)\n\n\ndef collect_pairs(root, dataset_name=\'o-hazy\'):\n    """Pair clear/hazy images across explicit split folders and common suffix conventions."""\n    dataset_name = normalize_dataset_name(dataset_name)\n    root = Path(root).resolve()\n    groups = {}\n    for p in sorted(root.rglob(\'*\')):\n        if not p.is_file() or p.suffix.lower() not in EXT:\n            continue\n        kind = _kind(p, root)\n        if kind is None:\n            continue\n        rel_parts = p.relative_to(root).parts\n        split = next((x.lower() for x in rel_parts if x.lower() in {\'train\', \'val\', \'validation\', \'test\'}), None)\n        if split == \'validation\': split = \'val\'\n        groups.setdefault(split or \'all\', {\'clear\': [], \'hazy\': []})[kind].append(p.resolve())\n    pairs = []\n    for split, sides in sorted(groups.items()):\n        exact, normalized = {}, {}\n        for clear in sides[\'clear\']:\n            for table, key in ((exact, clear.stem.lower()), (normalized, _base_stem(clear))):\n                if key in table:\n                    raise ValueError(f\'Trùng ảnh clear trong split {split}: {table[key]} và {clear}\')\n                table[key] = clear\n        used, ids = set(), set()\n        for hazy in sides[\'hazy\']:\n            clear = exact.get(hazy.stem.lower()) or normalized.get(_base_stem(hazy))\n            # RESIDE: one clear scene can have several atmospheric variants.\n            if clear is None and dataset_name in {\'reside6k\', \'sots-its\', \'sots-ots\'}:\n                clear = normalized.get(_base_stem(Path(hazy.stem.split(\'_\')[0])))\n            if clear is None:\n                raise ValueError(f\'Không ghép được ảnh hazy với clear trong split {split}: {hazy}\')\n            sid = hazy.stem.lower()\n            if sid in ids:\n                raise ValueError(f\'Trùng ID hazy trong split {split}: {sid}\')\n            ids.add(sid); used.add(clear)\n            pairs.append({\'id\': sid, \'scene\': _base_stem(clear), \'hazy\': str(hazy), \'clear\': str(clear),\n                          \'source_split\': None if split == \'all\' else split})\n        if set(sides[\'clear\']) != used:\n            raise ValueError(f\'Có ảnh clear thiếu hazy trong split {split}: {set(sides["clear"])-used}\')\n    if not pairs:\n        raise ValueError(f\'Không tìm thấy cặp clear/hazy trong {root}\')\n    if dataset_name == \'o-hazy\':\n        # Phase 3 protocol: IDs 1-40 for tuning, 41-45 held out for test.\n        def numeric(row):\n            m = re.match(r\'0*(\\d+)\', row[\'id\'])\n            return int(m.group(1)) if m else None\n        by_id = {}\n        for row in pairs:\n            sid = numeric(row)\n            if sid in by_id or sid is None:\n                raise ValueError(f\'Trùng hoặc thiếu ID scene O-HAZY: {row}\')\n            by_id[sid] = row\n        if set(by_id) != set(range(1, 46)):\n            raise ValueError(\'O-HAZY cần đủ 45 scene 01–45 để giữ protocol Phase 3.\')\n        pairs = [by_id[i] | {\'id\': i} for i in sorted(by_id)]\n    return pairs\n\n\ndef _holdout_train(rows, seed, fraction):\n    scenes = sorted({r[\'scene\'] for r in rows})\n    if len(scenes) < 2:\n        raise ValueError(\'Cần ít nhất hai scene train để tách validation.\')\n    random.Random(seed).shuffle(scenes)\n    val = set(scenes[:min(len(scenes)-1, max(1, round(len(scenes)*fraction)))])\n    return [r for r in rows if r[\'scene\'] not in val], [r for r in rows if r[\'scene\'] in val]\n\n\ndef partition(pairs, cfg):\n    dataset_name = normalize_dataset_name(cfg.get(\'dataset_name\', \'o-hazy\'))\n    if dataset_name == \'o-hazy\' and all(isinstance(p[\'id\'], int) for p in pairs):\n        val = set(cfg.get(\'val_ids\', [5, 15, 25, 35]))\n        if not val or not val <= set(range(1, 41)):\n            raise ValueError(\'val_ids O-HAZY phải thuộc 1–40 và không rỗng.\')\n        result = {\'train\': [p for p in pairs if p[\'id\'] <= 40 and p[\'id\'] not in val],\n                  \'val\': [p for p in pairs if p[\'id\'] in val],\n                  \'test\': [p for p in pairs if 41 <= p[\'id\'] <= 45]}\n    else:\n        explicit = {p.get(\'source_split\') for p in pairs}\n        if None in explicit and len(explicit) > 1:\n            raise ValueError(\'Không trộn dữ liệu phẳng với thư mục train/val/test.\')\n        if dataset_name.startswith(\'sots-\'):\n            # Some repackaged SOTS have train/val folders. They are still benchmark test images.\n            result = {\'train\': [], \'val\': [], \'test\': list(pairs)}\n        elif explicit == {None}:\n            raise ValueError(\'Dataset train cần thư mục train và test; không tự lấy test làm validation.\')\n        else:\n            result = {k: [p for p in pairs if p.get(\'source_split\') == k] for k in (\'train\', \'val\', \'test\')}\n            if result[\'train\'] and not result[\'val\']:\n                result[\'train\'], result[\'val\'] = _holdout_train(result[\'train\'], cfg.get(\'seed\', 2026), .1)\n    assert_disjoint(result)\n    return result\n\n\ndef assert_disjoint(splits):\n    seen = {}\n    for split, rows in splits.items():\n        ids = set()\n        for row in rows:\n            if row[\'id\'] in ids:\n                raise ValueError(f\'Trùng ID trong split {split}: {row["id"]}\')\n            ids.add(row[\'id\'])\n            keys = [(\'scene\', row[\'scene\']), (\'path\', row[\'clear\']), (\'path\', row[\'hazy\'])]\n            keys += [(\'clear_sha256\', row[\'clear_sha256\'])] if row.get(\'clear_sha256\') else []\n            keys += [(\'hazy_sha256\', row[\'hazy_sha256\'])] if row.get(\'hazy_sha256\') else []\n            for key in keys:\n                if key in seen and seen[key] != split:\n                    raise ValueError(f\'Rò rỉ dữ liệu giữa {seen[key]} và {split}: {key}\')\n                seen[key] = split\n\n\ndef dataset_splits(cfg):\n    name = normalize_dataset_name(cfg.get(\'dataset_name\', \'o-hazy\'))\n    data = Path(cfg.get(\'data_root\') or Path(cfg[\'project_root\']) / DATASET_SPECS[name][\'relative_root\']).expanduser().resolve()\n    if not data.is_dir():\n        raise FileNotFoundError(f\'Không có dataset {name} tại {data}. Đặt DATA_ROOT_OVERRIDE tới dữ liệu có sẵn.\')\n    splits = partition(collect_pairs(data, name), {**cfg, \'dataset_name\': name})\n    if cfg.get(\'train_root\'):\n        if not name.startswith(\'sots-\'):\n            raise ValueError(\'TRAIN_ROOT chỉ dành cho train ITS/OTS riêng khi đánh giá trên SOTS.\')\n        train_root = Path(cfg[\'train_root\']).expanduser().resolve()\n        if train_root == data or train_root.is_relative_to(data) or data.is_relative_to(train_root):\n            raise ValueError(\'TRAIN_ROOT phải tách biệt với thư mục benchmark SOTS.\')\n        training = collect_pairs(train_root, name)\n        explicit = {r.get(\'source_split\') for r in training}\n        if explicit == {None}:\n            splits[\'train\'], splits[\'val\'] = _holdout_train(training, cfg.get(\'seed\', 2026), .1)\n        elif explicit <= {\'train\', \'val\'} and \'train\' in explicit:\n            splits[\'train\'] = [r for r in training if r[\'source_split\'] == \'train\']\n            splits[\'val\'] = [r for r in training if r[\'source_split\'] == \'val\']\n            if not splits[\'val\']:\n                splits[\'train\'], splits[\'val\'] = _holdout_train(splits[\'train\'], cfg.get(\'seed\', 2026), .1)\n        else:\n            raise ValueError(\'TRAIN_ROOT chỉ chứa train/val hoặc các cặp ảnh phẳng, không chứa test.\')\n    requested = cfg.get(\'eval_only\')\n    eval_only = (not splits[\'train\']) if requested is None else bool(requested)\n    if not splits[\'test\'] or (not eval_only and (not splits[\'train\'] or not splits[\'val\'])):\n        raise ValueError(\'Cần test để đánh giá; fine-tune còn cần train và validation riêng. SOTS cần TRAIN_ROOT để train.\')\n    assert_disjoint(splits)\n    return data, splits, eval_only\n\n\ndef manifest_digest(splits):\n    rows = [{k: row[k] for k in (\'split\', \'id\', \'scene\', \'hazy_sha256\', \'clear_sha256\')}\n            for split in (\'train\', \'val\', \'test\') for row in splits[split]]\n    return hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()\n\n\ndef prepare(cfg):\n    work = Path(cfg[\'work_root\'])\n    work.mkdir(parents=True, exist_ok=True)\n    root, splits, evaluation_only = dataset_splits(cfg)\n    name = normalize_dataset_name(cfg.get(\'dataset_name\', \'o-hazy\'))\n    # Hash shared clear images once, including synthetic sets with many atmospheric variants.\n    hashes = {}\n    for split, rows in splits.items():\n        for row in rows:\n            row.update(dataset_name=name, split=split)\n            for key in (\'hazy\', \'clear\'):\n                if row[key] not in hashes:\n                    hashes[row[key]] = sha256(row[key])\n                row[key+\'_sha256\'] = hashes[row[key]]\n    assert_disjoint(splits)\n    value = {\'dataset_name\': name, \'data_root\': str(root), \'evaluation_only\': evaluation_only,\n             \'splits\': splits, \'manifest_sha256\': manifest_digest(splits)}\n    dump_json(work/\'inputs.json\', value)\n    write_csv(work/\'split_manifest.csv\', [r for rows in splits.values() for r in rows])\n    print(\'Dataset:\', name, {k: len(v) for k, v in splits.items()}, \'eval_only:\', evaluation_only, flush=True)\n    return value\n\n\ndef validate(cfg, inputs):\n    root, splits, only = dataset_splits(cfg)\n    if inputs[\'dataset_name\'] != normalize_dataset_name(cfg[\'dataset_name\']) or str(root) != inputs[\'data_root\']:\n        raise ValueError(\'Dataset/root đã đổi; chạy prepare lại.\')\n    if only != inputs[\'evaluation_only\']:\n        raise ValueError(\'Chế độ dataset đã đổi; chạy prepare lại.\')\n    for split in splits:\n        keys = lambda rows: [(r[\'id\'], r[\'hazy\'], r[\'clear\']) for r in rows]\n        if keys(splits[split]) != keys(inputs[\'splits\'][split]):\n            raise ValueError(\'Split đã đổi; chạy prepare lại.\')\n    checked = {}\n    for rows in inputs[\'splits\'].values():\n        for row in rows:\n            for key in (\'hazy\', \'clear\'):\n                if row[key] not in checked:\n                    checked[row[key]] = sha256(row[key])\n                if checked[row[key]] != row[key+\'_sha256\']:\n                    raise ValueError(f\'Dữ liệu đã đổi: {row[key]}\')\n    assert_disjoint(inputs[\'splits\'])\n    if manifest_digest(inputs[\'splits\']) != inputs[\'manifest_sha256\']:\n        raise ValueError(\'Manifest không còn khớp checksum.\')\n\n\ndef load_pair(pair, max_side):\n    images = []\n    for key in (\'hazy\', \'clear\'):\n        image = cv2.imread(pair[key], cv2.IMREAD_COLOR)\n        if image is None:\n            raise OSError(f\'Không đọc được {pair[key]}\')\n        images.append(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))\n    x, y = images\n    if (pair.get(\'dataset_name\') == \'sots-its\' and y.shape[:2] == (x.shape[0]+20, x.shape[1]+20)):\n        # Standard SOTS indoor GT includes a 10-pixel frame absent from the hazy image.\n        y = y[10:-10, 10:-10]\n    if x.shape != y.shape:\n        raise ValueError(f\'Cặp ảnh khác shape: scene {pair["id"]}; {x.shape}, {y.shape}\')\n    h, w = x.shape[:2]\n    if max_side and max(h, w) > max_side:\n        scale = max_side / max(h, w)\n        size = (round(w * scale), round(h * scale))\n        x, y = [cv2.resize(a, size, interpolation=cv2.INTER_AREA) for a in (x, y)]\n    return [torch.from_numpy(a.copy()).permute(2, 0, 1).float().div_(255) for a in (x, y)]\n\n\ndef pad32(x):\n    h, w = x.shape[-2:]\n    dh, dw = (-h) % 32, (-w) % 32\n    return F.pad(x, (0, dw, 0, dh), mode=\'reflect\' if dh < h and dw < w else \'replicate\')\n\n\ndef final_output(y):\n    return y[-1] if isinstance(y, (tuple, list)) else y\n\n\ndef metrics(pred, gt, border=4):\n    a, b = [t.detach().float().clamp(0, 1).cpu().numpy().transpose(1, 2, 0) for t in (pred, gt)]\n    if border:\n        a, b = [v[border:-border, border:-border] for v in (a, b)]\n    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)\n    psnr = float(-10 * np.log10(max(mse, 1e-12)))\n    # Same 11x11 Gaussian window/sigma and valid-region averaging as BasicSR SSIM.\n    ssim = float(structural_similarity(a, b, channel_axis=2, data_range=1.,\n                                      gaussian_weights=True, sigma=1.5, use_sample_covariance=False))\n    return psnr, ssim\n\n'
ARCH_SOURCE = '"""SwiftWave Dehaze Net: a small Haar encoder/decoder trained from random weights.\n\nThis is a new implementation combining established Haar and gated-convolution ideas;\nno claim of research novelty. It uses ordinary PyTorch operators only.\n"""\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\n\nARCH_VERSION = \'swdnet-v1\'\n\n\nclass HaarDown(nn.Module):\n    def forward(self, x):\n        a, b = x[..., ::2, ::2], x[..., ::2, 1::2]\n        c, d = x[..., 1::2, ::2], x[..., 1::2, 1::2]\n        return torch.cat(((a+b+c+d)*.5, (-a-b+c+d)*.5,\n                          (-a+b-c+d)*.5, (a-b-c+d)*.5), 1)\n\n\nclass HaarUp(nn.Module):\n    def forward(self, x):\n        ll, lh, hl, hh = x.chunk(4, 1)\n        # Stack in pixel-shuffle order, preserving each channel\'s four spatial samples.\n        a = (ll-lh-hl+hh)*.5\n        b = (ll-lh+hl-hh)*.5\n        c = (ll+lh-hl-hh)*.5\n        d = (ll+lh+hl+hh)*.5\n        return F.pixel_shuffle(torch.stack((a, b, c, d), 2).flatten(1, 2), 2)\n\n\nclass ChannelNorm(nn.Module):\n    def __init__(self, channels):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(1, channels, 1, 1))\n        self.bias = nn.Parameter(torch.zeros(1, channels, 1, 1))\n\n    def forward(self, x):\n        # Accumulate normalization in FP32, including during BF16 training.\n        z = x.float()\n        z = (z-z.mean(1, keepdim=True))*torch.rsqrt(z.var(1, keepdim=True, unbiased=False)+1e-6)\n        return (z*self.weight+self.bias).to(x.dtype)\n\n\nclass GatedWaveBlock(nn.Module):\n    def __init__(self, channels, dilation=1):\n        super().__init__()\n        self.norm = ChannelNorm(channels)\n        self.expand = nn.Conv2d(channels, 2*channels, 1)\n        self.spatial = nn.Conv2d(2*channels, 2*channels, 5, padding=2*dilation,\n                                 dilation=dilation, groups=2*channels)\n        self.context = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, channels, 1), nn.Sigmoid())\n        self.project = nn.Conv2d(channels, channels, 1)\n        self.scale = nn.Parameter(torch.full((1, channels, 1, 1), .1))\n\n    def forward(self, x):\n        a, b = self.spatial(self.expand(self.norm(x))).chunk(2, 1)\n        z = a*F.silu(b)\n        return x+self.scale*self.project(z*self.context(z))\n\n\ndef blocks(channels, count, dilated=False):\n    return nn.Sequential(*(GatedWaveBlock(channels, 1+(i % 2) if dilated else 1) for i in range(count)))\n\n\nclass SWDNet(nn.Module):\n    def __init__(self, width=32, depths=(2, 2, 4, 2, 2)):\n        super().__init__()\n        if width < 8 or len(depths) != 5 or any(d < 1 for d in depths):\n            raise ValueError(\'width >= 8; depths chứa năm số nguyên dương.\')\n        self.spec = {\'arch_version\': ARCH_VERSION, \'width\': int(width), \'depths\': list(depths)}\n        self.haar, self.inverse = HaarDown(), HaarUp()\n        self.stem = nn.Conv2d(12, width, 1)\n        self.enc1 = blocks(width, depths[0])  # All deep processing starts at half resolution.\n        self.down1 = nn.Conv2d(width*4, width*2, 1)\n        self.enc2 = blocks(width*2, depths[1])\n        self.down2 = nn.Conv2d(width*8, width*3, 1)\n        self.middle = blocks(width*3, depths[2], dilated=True)\n        self.up2 = nn.Conv2d(width*3, width*8, 1)\n        self.dec2 = blocks(width*2, depths[3])\n        self.up1 = nn.Conv2d(width*2, width*4, 1)\n        self.dec1 = blocks(width, depths[4])\n        self.wave_residual = nn.Conv2d(width, 12, 3, padding=1)\n        detail_width = max(8, width//2)\n        self.detail = nn.Sequential(nn.Conv2d(3, detail_width, 1), nn.SiLU(),\n                                    nn.Conv2d(detail_width, detail_width, 3, padding=1, groups=detail_width),\n                                    nn.SiLU(), nn.Conv2d(detail_width, 3, 1))\n        # Exact identity at initialization; only the two output heads start at zero.\n        nn.init.zeros_(self.wave_residual.weight); nn.init.zeros_(self.wave_residual.bias)\n        nn.init.zeros_(self.detail[-1].weight); nn.init.zeros_(self.detail[-1].bias)\n\n    def forward(self, x):\n        h, w = x.shape[-2:]\n        ph, pw = (-h) % 8, (-w) % 8\n        padded = F.pad(x, (0, pw, 0, ph), mode=\'replicate\') if ph or pw else x\n        e1 = self.enc1(self.stem(self.haar(padded)))\n        e2 = self.enc2(self.down1(self.haar(e1)))\n        z = self.middle(self.down2(self.haar(e2)))\n        z = self.dec2(self.inverse(self.up2(z))+e2)\n        z = self.dec1(self.inverse(self.up1(z))+e1)\n        restored = padded+self.inverse(self.wave_residual(z))+self.detail(padded)\n        # Never clamp inside training; the evaluator clamps to [0, 1].\n        return restored[..., :h, :w]\n'
RUNNER_SOURCE = '"""Phase 3.2: random initialization, resumable training, eager FP32 evaluation."""\nfrom pathlib import Path\nfrom collections import OrderedDict\nimport copy\nimport json\nimport math\nimport random\nimport re\nimport sys\nimport time\n\nimport cv2\nimport numpy as np\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\n\nimport data_utils as data\nfrom swdnet import SWDNet, HaarDown, HaarUp, ARCH_VERSION\n\n\ndef atomic_save(value, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix+\'.tmp\')\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef device_for(cfg):\n    device = torch.device(cfg.get(\'device\', \'cuda\'))\n    if device.type == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'Cần GPU CUDA cho notebook. CPU chỉ dành cho smoke test với device=cpu.\')\n    return device\n\n\ndef configure(cfg):\n    random.seed(cfg[\'seed\']); np.random.seed(cfg[\'seed\']); torch.manual_seed(cfg[\'seed\'])\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(cfg[\'seed\'])\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    cv2.setNumThreads(0)\n\n\ndef signature(cfg):\n    keys = (\'model\', \'seed\', \'schedule\', \'lr\', \'min_lr\', \'warmup_steps\', \'weight_decay\',\n            \'ema_decay\', \'l1_weight\', \'edge_weight\', \'train_max_side\', \'eval_max_side\',\n            \'crop_border\', \'val_every\', \'early_stop_patience\', \'early_stop_min_steps\', \'train_precision\')\n    return {k: cfg[k] for k in keys}\n\n\ndef validate_schedule(cfg):\n    if not cfg[\'schedule\'] or cfg[\'val_every\'] < 1 or cfg[\'cache_images\'] < 1:\n        raise ValueError(\'Cần schedule, val_every >= 1 và cache_images >= 1.\')\n    for stage in cfg[\'schedule\']:\n        if stage[\'steps\'] < 1 or stage[\'batch_size\'] < 1 or stage[\'crop\'] < 0:\n            raise ValueError(\'steps/batch_size phải dương; crop >= 0.\')\n        if stage[\'crop\'] == 0 and stage[\'batch_size\'] != 1:\n            raise ValueError(\'Giai đoạn ảnh đầy đủ cần batch_size=1 vì ảnh khác kích thước.\')\n\n\ndef learning_rate(cfg, step, total):\n    warm = min(cfg[\'warmup_steps\'], max(0, total-1))\n    if step <= warm:\n        return cfg[\'lr\']*step/max(warm, 1)\n    progress = (step-warm)/max(total-warm, 1)\n    return cfg[\'min_lr\']+.5*(cfg[\'lr\']-cfg[\'min_lr\'])*(1+math.cos(math.pi*progress))\n\n\ndef stage_at(schedule, step):\n    end = 0\n    for stage in schedule:\n        end += stage[\'steps\']\n        if step <= end:\n            return stage\n    raise ValueError(\'Step nằm ngoài schedule.\')\n\n\nclass PairCache:\n    def __init__(self, pairs, max_side, capacity):\n        self.pairs, self.max_side, self.capacity = pairs, max_side, capacity\n        self.cache = OrderedDict()\n\n    def sample(self, crop):\n        row = random.choice(self.pairs)\n        key = row[\'hazy\']\n        if key not in self.cache:\n            self.cache[key] = data.load_pair(row, self.max_side)\n            if len(self.cache) > self.capacity:\n                self.cache.popitem(last=False)\n        self.cache.move_to_end(key)\n        x, y = self.cache[key]\n        if crop:\n            h, w = x.shape[-2:]\n            if h < crop or w < crop:\n                x, y = [F.pad(v, (0, max(0, crop-w), 0, max(0, crop-h)), mode=\'replicate\') for v in (x, y)]\n            h, w = x.shape[-2:]\n            top, left = random.randrange(h-crop+1), random.randrange(w-crop+1)\n            x, y = [v[:, top:top+crop, left:left+crop] for v in (x, y)]\n            if random.random() < .5:\n                x, y = x.flip(-1), y.flip(-1)\n            if random.random() < .5:\n                x, y = x.flip(-2), y.flip(-2)\n            turns = random.randrange(4)\n            x, y = x.rot90(turns, (-2, -1)), y.rot90(turns, (-2, -1))\n        return x, y\n\n    def batch(self, crop, batch_size, device):\n        pairs = [self.sample(crop) for _ in range(batch_size)]\n        return [torch.stack([p[k] for p in pairs]).to(device) for k in (0, 1)]\n\n\ndef restoration_loss(pred, gt, cfg):\n    pred, gt = pred.float(), gt.float()\n    mse = F.mse_loss(pred, gt)\n    l1 = F.l1_loss(pred, gt)\n    # First differences, no external perceptual network or pretrained teacher.\n    edge = (F.l1_loss(pred[..., 1:, :]-pred[..., :-1, :], gt[..., 1:, :]-gt[..., :-1, :])\n            + F.l1_loss(pred[..., :, 1:]-pred[..., :, :-1], gt[..., :, 1:]-gt[..., :, :-1]))*.5\n    return mse+cfg[\'l1_weight\']*l1+cfg[\'edge_weight\']*edge\n\n\n@torch.no_grad()\ndef update_ema(ema, model, step, decay):\n    decay = min(decay, (1+step)/(10+step))\n    for target, source in zip(ema.parameters(), model.parameters()):\n        target.lerp_(source, 1-decay)\n    for target, source in zip(ema.buffers(), model.buffers()):\n        target.copy_(source)\n\n\n@torch.no_grad()\ndef quality(model, pairs, cfg, device, predictions=None):\n    model.eval()\n    rows = []\n    for row in pairs:\n        x, gt = data.load_pair(row, cfg[\'eval_max_side\'])\n        pred = model(x[None].to(device))[0]\n        if not torch.isfinite(pred).all():\n            raise FloatingPointError(f\'Output không hữu hạn: {row["id"]}\')\n        pred = pred.clamp(0, 1).cpu()\n        psnr, ssim = data.metrics(pred, gt, cfg[\'crop_border\'])\n        rows.append({\'dataset\': cfg[\'dataset_name\'], \'id\': row[\'id\'], \'PSNR\': psnr, \'SSIM\': ssim,\n                     \'height\': x.shape[-2], \'width\': x.shape[-1]})\n        if predictions:\n            folder = Path(predictions); folder.mkdir(parents=True, exist_ok=True)\n            image = (pred.permute(1, 2, 0).numpy()*255).round().astype(np.uint8)\n            name = re.sub(r\'[^\\w.-]\', \'_\', str(row[\'id\']))\n            cv2.imwrite(str(folder/(name+\'.png\')), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))\n    return rows\n\n\ndef train(cfg, inputs, output, resume=False):\n    if inputs[\'evaluation_only\']:\n        raise ValueError(\'SOTS chuẩn không có train. Cần TRAIN_ROOT ITS/OTS riêng hoặc benchmark checkpoint đã train.\')\n    validate_schedule(cfg)\n    last, best_path = output/\'last.pth\', output/\'best.pth\'\n    if not resume and (last.exists() or best_path.exists()):\n        raise FileExistsError(\'Run đã có checkpoint. Đổi RUN_NAME để train từ đầu, hoặc RUN_MODE=resume để tiếp tục.\')\n    device = device_for(cfg)\n    model = SWDNet(**cfg[\'model\']).to(device)\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg[\'lr\'], weight_decay=cfg[\'weight_decay\'])\n    total = sum(s[\'steps\'] for s in cfg[\'schedule\'])\n    start, best, stale, history, completed = 0, -math.inf, 0, [], False\n    arch_sha = data.sha256(Path(__file__).with_name(\'swdnet.py\'))\n    train_dtype = (torch.bfloat16 if device.type == \'cuda\' and cfg[\'train_precision\'] == \'bf16\'\n                   and torch.cuda.is_bf16_supported() else None)\n    if resume:\n        saved = torch.load(last, map_location=device, weights_only=False)\n        if (saved[\'train_signature\'] != signature(cfg) or saved[\'manifest_sha256\'] != inputs[\'manifest_sha256\']\n                or saved[\'architecture_sha256\'] != arch_sha or saved[\'initialization\'] != \'random\'):\n            raise ValueError(\'Checkpoint resume không khớp kiến trúc, lịch train hoặc manifest dataset.\')\n        if saved[\'actual_train_precision\'] != (\'BF16\' if train_dtype else \'FP32\'):\n            raise ValueError(\'Precision của runtime resume khác runtime training trước.\')\n        model.load_state_dict(saved[\'params\']); ema.load_state_dict(saved[\'params_ema\'])\n        optimizer.load_state_dict(saved[\'optimizer\'])\n        start, best, stale, history = saved[\'step\'], saved[\'best_val_PSNR\'], saved[\'stale\'], saved[\'history\']\n        completed = saved[\'completed\']\n        random.setstate(saved[\'random_state\']); np.random.set_state(saved[\'numpy_state\'])\n        torch.set_rng_state(saved[\'torch_state\'].cpu())\n        if device.type == \'cuda\':\n            torch.cuda.set_rng_state_all([s.cpu() for s in saved[\'cuda_states\']])\n    cache = PairCache(inputs[\'splits\'][\'train\'], cfg[\'train_max_side\'], cfg[\'cache_images\'])\n    print(f\'SWDNet: random initialization; device={device}; precision={"BF16" if train_dtype else "FP32"}; \'\n          f\'parameters={sum(p.numel() for p in model.parameters()):,}; start={start}/{total}\', flush=True)\n    started = time.perf_counter()\n    step = start\n\n    def base_state():\n        return {\'arch_version\': ARCH_VERSION, \'architecture_sha256\': arch_sha, \'model_spec\': cfg[\'model\'],\n                \'dataset_name\': cfg[\'dataset_name\'], \'initialization\': \'random\',\n                \'manifest_sha256\': inputs[\'manifest_sha256\'], \'train_signature\': signature(cfg),\n                \'step\': step, \'best_val_PSNR\': best, \'eval_max_side\': cfg[\'eval_max_side\'],\n                \'crop_border\': cfg[\'crop_border\'], \'actual_train_precision\': \'BF16\' if train_dtype else \'FP32\',\n                \'params_ema\': {k: v.detach().cpu().clone() for k, v in ema.state_dict().items()}}\n\n    if not completed:\n        for step in range(start+1, total+1):\n            stage = stage_at(cfg[\'schedule\'], step)\n            lr = learning_rate(cfg, step, total)\n            for group in optimizer.param_groups:\n                group[\'lr\'] = lr\n            model.train()\n            x, gt = cache.batch(stage[\'crop\'], stage[\'batch_size\'], device)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, dtype=train_dtype or torch.bfloat16, enabled=train_dtype is not None):\n                pred = model(x)\n                loss = restoration_loss(pred, gt, cfg)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f\'Loss không hữu hạn tại step {step}; giữ checkpoint last gần nhất.\')\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)\n            optimizer.step()\n            update_ema(ema, model, step, cfg[\'ema_decay\'])\n            if step % cfg[\'log_every\'] == 0:\n                print(f\'step={step}/{total} crop={stage["crop"]} loss={loss.item():.6f} lr={lr:.2e}\', flush=True)\n            if step % cfg[\'val_every\'] == 0 or step == total:\n                rows = quality(ema, inputs[\'splits\'][\'val\'], cfg, device)\n                score = float(np.mean([r[\'PSNR\'] for r in rows]))\n                if not math.isfinite(score):\n                    raise FloatingPointError(\'Validation PSNR không hữu hạn.\')\n                improved = score > best\n                if improved:\n                    best, stale = score, 0\n                elif step >= cfg[\'early_stop_min_steps\']:\n                    stale += 1\n                completed = step == total or (cfg[\'early_stop_patience\'] > 0 and stale >= cfg[\'early_stop_patience\'])\n                history.append({\'step\': step, \'crop\': stage[\'crop\'], \'loss\': loss.item(), \'lr\': lr,\n                                \'val_PSNR\': score, \'best_val_PSNR\': best, \'elapsed_s_this_session\': time.perf_counter()-started})\n                data.write_csv(output/\'history.csv\', history)\n                if improved:\n                    atomic_save({**base_state(), \'kind\': \'best\'}, best_path)\n                    data.write_csv(output/\'best_val_metrics.csv\', rows)\n                atomic_save({**base_state(), \'kind\': \'last\', \'params\': model.state_dict(),\n                             \'optimizer\': optimizer.state_dict(), \'history\': history, \'stale\': stale, \'completed\': completed,\n                             \'random_state\': random.getstate(), \'numpy_state\': np.random.get_state(),\n                             \'torch_state\': torch.get_rng_state(),\n                             \'cuda_states\': torch.cuda.get_rng_state_all() if device.type == \'cuda\' else []}, last)\n                print(f\'validation={score:.4f} dB; best={best:.4f}; checkpoint saved\', flush=True)\n                if completed:\n                    break\n    return {\'initialization\': \'random\', \'steps\': step, \'best_val_PSNR\': best,\n            \'seconds_this_session\': time.perf_counter()-started, \'resumed\': resume}\n\n\n@torch.no_grad()\ndef profile_gflops(model, x):\n    ops = {\'conv_linear\': 0, \'haar\': 0}\n    handles = []\n    states = {m: m.training for m in model.modules()}\n\n    def hook(module, args, y):\n        if isinstance(module, nn.Conv2d):\n            ops[\'conv_linear\'] += 2*y.numel()*(module.in_channels//module.groups)*math.prod(module.kernel_size)\n        elif isinstance(module, nn.Linear):\n            ops[\'conv_linear\'] += 2*args[0].numel()*module.out_features\n        elif isinstance(module, (HaarDown, HaarUp)):\n            ops[\'haar\'] += 4*args[0].numel()  # Three additions and one multiply per output coefficient.\n    try:\n        model.eval()\n        for module in model.modules():\n            handles.append(module.register_forward_hook(hook))\n        model(x)\n    finally:\n        for handle in handles:\n            handle.remove()\n        for module, training in states.items():\n            module.training = training\n    return {\'counted\': ops[\'conv_linear\']/1e9, \'supplemental\': ops[\'haar\']/1e9,\n            \'estimated_total\': sum(ops.values())/1e9}\n\n\n@torch.no_grad()\ndef latency(model, x, cfg):\n    model.eval()\n    for _ in range(cfg[\'warmup\']):\n        model(x)\n    values = []\n    if x.is_cuda:\n        torch.cuda.synchronize(x.device)\n        torch.cuda.reset_peak_memory_stats(x.device)\n    for _ in range(cfg[\'timing_repeats\']):\n        if x.is_cuda:\n            start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)\n            start.record(); model(x); end.record(); end.synchronize()\n            values.append(start.elapsed_time(end))\n        else:\n            start = time.perf_counter(); model(x); values.append((time.perf_counter()-start)*1000)\n    if not values:\n        raise ValueError(\'timing_repeats phải lớn hơn 0.\')\n    mean = float(np.mean(values))\n    return {\'latency_ms\': mean, \'FPS\': 1000/mean, \'p50_ms\': float(np.median(values)),\n            \'p95_ms\': float(np.percentile(values, 95)),\n            \'peak_allocated_GiB\': torch.cuda.max_memory_allocated(x.device)/2**30 if x.is_cuda else None}\n\n\ndef evaluate(cfg, inputs, output):\n    path = Path(cfg.get(\'evaluation_checkpoint\') or output/\'best.pth\').expanduser().resolve()\n    if not path.is_file():\n        raise FileNotFoundError(f\'Cần checkpoint SWDNet đã train: {path}. Không đánh giá trọng số ngẫu nhiên trên SOTS.\')\n    saved = torch.load(path, map_location=\'cpu\', weights_only=False)\n    if saved[\'arch_version\'] != ARCH_VERSION or saved[\'initialization\'] != \'random\' or saved[\'step\'] < 1:\n        raise ValueError(\'Checkpoint không thuộc kiến trúc SWDNet train từ đầu.\')\n    if saved[\'architecture_sha256\'] != data.sha256(Path(__file__).with_name(\'swdnet.py\')):\n        raise ValueError(\'Mã kiến trúc khác checkpoint; dùng đúng phiên bản swdnet.py.\')\n    cross_dataset = saved[\'dataset_name\'] != cfg[\'dataset_name\']\n    if cross_dataset and not inputs[\'evaluation_only\']:\n        raise ValueError(\'Checkpoint thuộc dataset khác; chỉ dùng cross-dataset khi eval_only=True.\')\n    if not cross_dataset and saved[\'manifest_sha256\'] != inputs[\'manifest_sha256\']:\n        raise ValueError(\'Manifest dataset khác checkpoint training.\')\n    device = device_for(cfg)\n    model = SWDNet(**saved[\'model_spec\']).to(device).eval()\n    model.load_state_dict(saved[\'params_ema\'], strict=True)\n    # Save quality and partial summary before timing, so a benchmark failure does not lose PSNR.\n    rows = quality(model, inputs[\'splits\'][\'test\'], cfg, device, output/\'predictions\')\n    data.write_csv(output/\'test_metrics.csv\', rows)\n    summary = {\'dataset\': cfg[\'dataset_name\'], \'model\': \'SWDNet\', \'source_dataset\': saved[\'dataset_name\'],\n               \'device\': str(device), \'precision\': \'FP32\', \'execution\': \'eager\', \'batch_size\': 1,\n               \'images\': len(rows), \'eval_max_side\': cfg[\'eval_max_side\'], \'crop_border\': cfg[\'crop_border\'],\n               \'PSNR\': float(np.mean([r[\'PSNR\'] for r in rows])), \'SSIM\': float(np.mean([r[\'SSIM\'] for r in rows])),\n               \'params_M\': sum(p.numel() for p in model.parameters())/1e6}\n    data.dump_json(output/\'quality_partial.json\', summary)\n    timing_rows = []\n    for pair in inputs[\'splits\'][\'test\']:\n        x, _ = data.load_pair(pair, cfg[\'eval_max_side\'])\n        probe = x[None].to(device)\n        timing_rows.append({\'id\': pair[\'id\'], \'height\': x.shape[-2], \'width\': x.shape[-1],\n                            **latency(model, probe, cfg), **profile_gflops(model, probe)})\n    probe = torch.zeros(1, 3, 256, 256, device=device)\n    f256, t256 = profile_gflops(model, probe), latency(model, probe, cfg)\n    avg_ms = float(np.mean([r[\'latency_ms\'] for r in timing_rows]))\n    summary.update(latency_ms=avg_ms, FPS=1000/avg_ms, FPS_256=t256[\'FPS\'],\n                   GFLOPs_256_counted=f256[\'counted\'], GFLOPs_256_supplemental=f256[\'supplemental\'],\n                   GFLOPs_256_estimated_total=f256[\'estimated_total\'],\n                   GFLOPs_test_estimated_mean=float(np.mean([r[\'estimated_total\'] for r in timing_rows])))\n    data.write_csv(output/\'test_timing_gflops.csv\', timing_rows)\n    data.dump_json(output/\'probe_256.json\', {\'gflops\': f256, \'timing\': t256})\n    data.write_csv(output/\'summary.csv\', [summary])\n    baseline = []\n    for pair in inputs[\'splits\'][\'test\']:\n        x, gt = data.load_pair(pair, cfg[\'eval_max_side\'])\n        psnr, ssim = data.metrics(x, gt, cfg[\'crop_border\'])\n        baseline.append({\'id\': pair[\'id\'], \'PSNR\': psnr, \'SSIM\': ssim})\n    data.write_csv(output/\'hazy_input_metrics.csv\', baseline)\n    metadata = {\'status\': \'MEASURED\', \'initialization\': \'random\', \'source_dataset\': saved[\'dataset_name\'],\n                \'cross_dataset\': cross_dataset, \'checkpoint\': str(path), \'checkpoint_sha256\': data.sha256(path),\n                \'selected_step\': saved[\'step\'], \'best_val_PSNR\': saved[\'best_val_PSNR\'],\n                \'manifest_sha256\': inputs[\'manifest_sha256\'], \'architecture\': saved[\'model_spec\'],\n                \'architecture_sha256\': saved[\'architecture_sha256\'], \'torch\': torch.__version__,\n                \'gpu\': torch.cuda.get_device_name(device) if device.type == \'cuda\' else \'CPU smoke test\',\n                \'cuda\': torch.version.cuda, \'train_precision\': saved[\'actual_train_precision\'],\n                \'GFLOPs_method\': \'1 MAC=2 FLOPs; Conv/Linear counted; Haar arithmetic supplemental; normalization, gates, activations, pooling, adds excluded\',\n                \'FPS_scope\': \'batch 1, FP32 eager; model forward including internal pad/crop; excludes I/O, transfer, metrics and warmup\'}\n    data.dump_json(output/\'metadata.json\', metadata)\n    lines = [\'# Phase 3.2 — SWDNet train từ đầu\', \'\',\n             \'| Dataset | Model | PSNR | SSIM | FPS | FPS 256 | GFLOPs counted | Haar bổ sung | GFLOPs tổng ước tính | Params M |\',\n             \'|---|---|---:|---:|---:|---:|---:|---:|---:|---:|\',\n             f"| {cfg[\'dataset_name\']} | SWDNet | {summary[\'PSNR\']:.4f} | {summary[\'SSIM\']:.5f} | {summary[\'FPS\']:.2f} | {summary[\'FPS_256\']:.2f} | {f256[\'counted\']:.3f} | {f256[\'supplemental\']:.3f} | {f256[\'estimated_total\']:.3f} | {summary[\'params_M\']:.3f} |", \'\',\n             f"Input hazy: PSNR={np.mean([r[\'PSNR\'] for r in baseline]):.4f}, SSIM={np.mean([r[\'SSIM\'] for r in baseline]):.5f}.",\n             f"Checkpoint: step {saved[\'step\']}; validation PSNR={saved[\'best_val_PSNR\']:.4f}; nguồn train={saved[\'dataset_name\']}.",\n             f"Thiết bị: {metadata[\'gpu\']}; train {saved[\'actual_train_precision\']}, đánh giá FP32 eager.",\n             \'GFLOPs là số phép tính mỗi ảnh; normalization/gating/activation/pooling/elementwise không nằm trong số đếm.\',\n             \'Không suy ra FPS từ GFLOPs; cần đo các baseline trên cùng GPU và evaluator trước khi kết luận tăng tốc.\',\n             \'Khởi tạo ngẫu nhiên, không teacher/KD/pretrained; validation chọn EMA checkpoint, test chỉ lập báo cáo.\',\n             \'O-HAZY chỉ có 36 train/4 val/5 test trong protocol này; test các phase trước đã được xem.\',\n             \'MEASURED không có nghĩa là đã đạt PSNR/FPS của WDMamba pretrained.\']\n    (output/\'final_report.md\').write_text(\'\\n\'.join(lines)+\'\\n\', encoding=\'utf-8\')\n    print(json.dumps(summary, indent=2), flush=True)\n    return summary\n\n\ndef run(cfg, mode):\n    if mode in (\'train\', \'resume\') and cfg.get(\'evaluation_checkpoint\'):\n        raise ValueError(\'EVALUATION_CHECKPOINT chỉ dùng trong benchmark; để trống khi train/resume.\')\n    configure(cfg)\n    inputs = json.loads((Path(cfg[\'work_root\'])/\'inputs.json\').read_text(encoding=\'utf-8\'))\n    data.validate(cfg, inputs)\n    output = Path(cfg[\'output_root\']); output.mkdir(parents=True, exist_ok=True)\n    if mode in (\'train\', \'resume\'):\n        info = train(cfg, inputs, output, resume=mode == \'resume\')\n        data.dump_json(output/\'training.json\', info)\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n    elif mode != \'benchmark\':\n        raise ValueError(\'Chọn train, resume hoặc benchmark.\')\n    data.dump_json(output/\'config.json\', cfg)\n    return evaluate(cfg, inputs, output)\n\n\nif __name__ == \'__main__\':\n    config = json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))\n    if sys.argv[2] == \'prepare\':\n        data.prepare(config)\n    else:\n        run(config, sys.argv[2])\n'

import ast
for source in (DATA_SOURCE, ARCH_SOURCE, RUNNER_SOURCE): ast.parse(source)
(RUN_ROOT/'data_utils.py').write_text(DATA_SOURCE, encoding='utf-8')
(RUN_ROOT/'swdnet.py').write_text(ARCH_SOURCE, encoding='utf-8')
(RUN_ROOT/'train_scratch.py').write_text(RUNNER_SOURCE, encoding='utf-8')
sys.path.insert(0, str(RUN_ROOT))
print('Embedded Phase 3.2 sources: PASS')
smoke = "import torch; from swdnet import SWDNet; m=SWDNet().cuda(); x=torch.rand(1,3,65,71,device='cuda'); y=m(x); y.square().mean().backward(); assert y.shape==x.shape and torch.isfinite(y).all(); print('GPU forward/backward: PASS')"
command([PYTHON, '-c', smoke], cwd=RUN_ROOT)


## 3. Manifest, train từ đầu và đánh giá

`train` luôn yêu cầu output run mới; `resume` tiếp tục `last.pth` sau gián đoạn; `benchmark` chỉ nạp
`best.pth`. Mỗi checkpoint lưu architecture hash, manifest hash, seed, precision và lịch train.

In [ ]:
def run_phase(mode):
    CONFIG_PATH = RUN_ROOT/'config.json'
    command([PYTHON, '-u', RUN_ROOT/'train_scratch.py', CONFIG_PATH, mode], log=mode+'.log')
run_phase('prepare')
if RUN_MODE not in ('train', 'resume', 'benchmark'):
    raise ValueError('RUN_MODE phải là train, resume hoặc benchmark')
run_phase(RUN_MODE)


## 4. Đọc báo cáo và đóng gói

`summary.csv` ghi PSNR, SSIM, FPS, FPS 256, GFLOPs counted, GFLOPs Haar bổ sung, GFLOPs tổng và số tham số.
GFLOPs là phép tính mỗi ảnh, không phải GFLOP/s; normalization, activation, pooling và cộng elementwise
được nêu riêng và không tính vào counted.

In [ ]:
from IPython.display import Markdown, display
display(Markdown((OUTPUT_ROOT/'final_report.md').read_text(encoding='utf-8')))


In [ ]:
import shutil
for name in ('swdnet.py', 'data_utils.py', 'train_scratch.py', 'inputs.json', 'split_manifest.csv',
             'prepare.log', RUN_MODE+'.log'):
    source = RUN_ROOT/name
    if source.is_file(): shutil.copy2(source, OUTPUT_ROOT/name)
archive = shutil.make_archive(str(RUN_ROOT/'phase3_2_swdnet_results'), 'zip', OUTPUT_ROOT)
print('Kết quả:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass
